# STARK-Amazon Data Exploration — QA Dataset

Inspects splits, queries, and corpus statistics using only the QA dataset.
**Runs on Mac M2.** Does not load the SKB (which requires ~19 GB RAM).
See `01b_data_exploration_skb.ipynb` for product-node and graph-edge inspection (aixsrv1 only).

## 1. Load QA dataset

In [ ]:
from stark_qa import load_qa

qa_dataset = load_qa('amazon')
print(f'QA dataset size: {len(qa_dataset)}')

## 2. Inspect the official splits

In [ ]:
idx_split = qa_dataset.get_idx_split()
print('Available splits:', list(idx_split.keys()))

for split_name, ids in idx_split.items():
    print(f'  {split_name:12s}: {len(ids):,} queries')

In [ ]:
# Split indices are torch.tensor objects — convert to int immediately
test_ids    = [int(i) for i in idx_split['test']]      # full test set (1,642 queries) — Step B metrics
test_01_ids = [int(i) for i in idx_split['test-0.1']]  # fixed official split (164 queries) — Step C and ablations

print(f'test     : {len(test_ids):,} queries')
print(f'test-0.1 : {len(test_01_ids):,} queries')

assert set(test_01_ids).issubset(set(test_ids)), 'test-0.1 is NOT a subset of test!'
print('Confirmed: test-0.1 is a subset of test')

## 3. Inspect a single query

In [ ]:
# qa_dataset[idx] returns a 4-tuple: (query, query_id, answer_ids, meta)
query, _, answer_ids, _ = qa_dataset[test_ids[0]]

print(f'Query:\n  {query}')
suffix = '...' if len(answer_ids) > 5 else ''
print(f'Answer node IDs ({len(answer_ids)}): {answer_ids[:5]}{suffix}')

In [ ]:
# Look at a few more queries to get a feel for the distribution
for i in range(5):
    q, _, ans, _ = qa_dataset[test_ids[i]]
    print(f'[{i}] answers={len(ans):2d} | {q[:100]}')

## 4. Quick corpus statistics

In [ ]:
import numpy as np

# answer_ids is at index [2] in the 4-tuple
answer_sizes = [len(qa_dataset[i][2]) for i in test_ids]
print(f'Answer set size — min: {min(answer_sizes)}, max: {max(answer_sizes)}, '
      f'mean: {np.mean(answer_sizes):.1f}, median: {np.median(answer_sizes):.0f}')

In [ ]:
# query is still at index [0]
query_lengths = [len(qa_dataset[i][0].split()) for i in test_ids]
print(f'Query length (words) — min: {min(query_lengths)}, max: {max(query_lengths)}, '
      f'mean: {np.mean(query_lengths):.1f}')